# MMMU-clean → MMMU-Pro Held-out-500
## Multimodal Academic Subject / Domain Classifier

**Model:** `Qwen/Qwen3-VL-2B-Instruct`  
**Training:** 4-bit QLoRA + trainable 30-way classification head  
**Platform:** Kaggle single NVIDIA GPU (P100/T4/A10/A100 compatible path)  
**Primary evaluation:** frozen 500-item MMMU-Pro cohort

### Scientific design

We train a **30-subject multimodal classifier** rather than a direct six-domain
classifier. The predicted subject is deterministically mapped to one of the
project's six domains. This guarantees hierarchical consistency and provides
both fine-grained subject accuracy and coarse-grained domain accuracy.

The model receives:

- the complete question stem;
- every textual answer option;
- all referenced images, including images embedded in answer options;
- multiple images in their original semantic roles.

It **never receives** the gold answer, explanation, or CoT.

### Leakage control

MMMU-Pro is derived from MMMU. Before training, every MMMU `dev`,
`validation`, and `test` item is audited against the full 1,730-item MMMU-Pro
`standard (10 options)` test source.

A source MMMU item is removed if any deterministic exact-overlap signal is
present:

1. exact normalized item ID;
2. exact conservatively normalized question;
3. at least one pixel-identical visual after EXIF orientation, RGB conversion,
   and canonical RGB hashing.

This is intentionally conservative. Semantic near-duplicates are not silently
removed by an arbitrary threshold; exact-cleaning statistics are archived.

### Multi-image / image-option strategy

Images are not collapsed into a montage. Instead, each unique physical image is
inserted at its first semantic occurrence:

- inside the question when referenced there;
- inside the corresponding option when referenced there;
- otherwise under `Additional source visual`.

Repeated references to the same physical image are represented textually after
the first visual insertion. A per-sample visual-token budget is divided across
all unique images so multi-image examples remain trainable on Kaggle.

In [1]:
# ================================================================
# 1. Environment installation — Kaggle-safe image stack bootstrap
# ================================================================
import sys
import subprocess
import importlib
import importlib.util
import importlib.metadata

PINNED = {
    "transformers": "5.16.1",
    "peft": "0.20.0",
    "bitsandbytes": "0.49.0",
    "Pillow": "11.3.0",
}

# Only these runtime-critical packages are forced to exact versions.
critical_specs = [
    "transformers==5.16.1",
    "peft==0.20.0",
    "bitsandbytes==0.49.0",
    "Pillow==11.3.0",
]

# Non-critical packages are installed only when their import is absent.
# Bounds avoid needlessly upgrading Kaggle to pandas 3.x / sklearn 1.9+.
OPTIONAL_REQUIREMENTS = {
    "accelerate": "accelerate>=1.7",
    "datasets": "datasets>=3.0",
    "huggingface_hub": "huggingface_hub>=0.24",
    "pandas": "pandas>=2.0,<2.4",
    "pyarrow": "pyarrow>=15",
    "sklearn": "scikit-learn>=1.4,<1.9",
    "tqdm": "tqdm>=4.66",
}

def installed_version(distribution_name):
    try:
        return importlib.metadata.version(distribution_name)
    except importlib.metadata.PackageNotFoundError:
        return None


def purge_pil_modules():
    for module_name in list(sys.modules):
        if module_name == "PIL" or module_name.startswith("PIL."):
            sys.modules.pop(module_name, None)
    importlib.invalidate_caches()


def pillow_integrity_smoke_test():
    """
    Functional Pillow smoke test.

    Do not inspect private symbols such as PIL._typing._Ink: those are
    version-internal implementation details and are not a stable API.

    The real requirement for this project is that Pillow's drawing stack
    imports and executes successfully, because torchvision/Transformers
    depend on that import chain.
    """
    from PIL import Image, ImageDraw, ImageFont

    canvas = Image.new("RGB", (16, 16), "white")
    drawer = ImageDraw.Draw(canvas)
    drawer.text(
        (1, 1),
        "x",
        fill="black",
        font=ImageFont.load_default(),
    )

    # Force a simple pixel access so the created image is genuinely usable.
    _ = canvas.getpixel((0, 0))
    return True

critical_to_install = [
    spec
    for dist, required in PINNED.items()
    for spec in [f"{dist}=={required}"]
    if installed_version(dist) != required
]

optional_to_install = []
for module_name, spec in OPTIONAL_REQUIREMENTS.items():
    try:
        importlib.util.find_spec(module_name)
        present = importlib.util.find_spec(module_name) is not None
    except Exception:
        present = False
    if not present:
        optional_to_install.append(spec)

install_specs = critical_to_install + optional_to_install

if install_specs:
    print("Installing only required packages:")
    for spec in install_specs:
        print("  -", spec)
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-cache-dir",
            *install_specs,
        ]
    )
else:
    print("Required packages already available; no broad environment upgrade.")

try:
    pillow_integrity_smoke_test()
except Exception as first_error:
    print(
        "Pillow integrity check failed; repairing exact Pillow 11.3.0 installation."
    )
    print("Initial Pillow error:", repr(first_error))

    purge_pil_modules()

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--force-reinstall",
            "--no-deps",
            "--no-cache-dir",
            "Pillow==11.3.0",
        ]
    )

    purge_pil_modules()
    pillow_integrity_smoke_test()

print("Pinned package versions:")
for dist, required in PINNED.items():
    observed = installed_version(dist)
    print(f"  {dist}: {observed}")
    if observed != required:
        raise RuntimeError(
            f"{dist} version mismatch: {observed} != {required}"
        )

# Verify the exact import chain that previously failed:
# transformers -> torchvision -> PIL.
# Keep Kaggle's native torch/torchvision pair untouched.
try:
    import torch
    import torchvision
    from torchvision.io import ImageReadMode
except Exception as exc:
    raise RuntimeError(
        "Kaggle torch/torchvision image-stack preflight failed after Pillow repair. "
        "Do not independently upgrade torch or torchvision. "
        f"torch={installed_version('torch')}, "
        f"torchvision={installed_version('torchvision')}, "
        f"Pillow={installed_version('Pillow')}. "
        f"Underlying error: {exc!r}"
    ) from exc

print(
    "Image-stack preflight PASS | "
    f"torch={installed_version('torch')} | "
    f"torchvision={installed_version('torchvision')} | "
    f"Pillow={installed_version('Pillow')}"
)

Installing only required packages:
  - transformers==5.16.1
  - peft==0.20.0
  - bitsandbytes==0.49.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 336.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 341.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 234.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 366.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 259.5 MB/s eta 0:00:00
Pinned package versions:
  transformers: 5.16.1
  peft: 0.20.0
  bitsandbytes: 0.49.0
  Pillow: 11.3.0
Image-stack preflight PASS | torch=2.10.0+cu128 | torchvision=0.25.0+cu128 | Pillow=11.3.0


### Kaggle image-stack preflight

This notebook pins `Pillow==11.3.0` and performs an import-level integrity test
before Transformers is imported. It specifically guards against a mixed or partially upgraded Pillow installation by exercising the actual image-drawing import path used downstream.

PyTorch and torchvision are intentionally left at Kaggle's native versions so
their compiled CUDA/operator compatibility is not disturbed.

In [2]:
# ================================================================
# 2. Imports, deterministic runtime, GPU policy
# ================================================================
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import ast as py_ast
import gc
import hashlib
import io
import json
import math
import random
import re
import time
import unicodedata
from collections import Counter, defaultdict
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from datasets import load_dataset
from huggingface_hub import snapshot_download
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from tqdm.auto import tqdm

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("A Kaggle GPU session is required.")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_CC = torch.cuda.get_device_capability(0)
GPU_MAJOR = GPU_CC[0]

# BF16 hardware support starts at NVIDIA compute capability 8.0 (Ampere).
# PyTorch may report bf16 software availability on older GPUs, so the
# architecture gate is authoritative for this notebook.
USE_BF16 = bool(
    GPU_MAJOR >= 8
    and torch.cuda.is_bf16_supported()
)
COMPUTE_DTYPE = (
    torch.bfloat16
    if USE_BF16
    else torch.float16
)
ATTN_IMPLEMENTATION = (
    "sdpa"
    if GPU_MAJOR >= 7
    else "eager"
)

print("GPU:", GPU_NAME)
print("Compute capability:", GPU_CC)
print("Compute dtype:", COMPUTE_DTYPE)
print("BF16 hardware gate:", "enabled" if USE_BF16 else "disabled")
print("Attention implementation:", ATTN_IMPLEMENTATION)

GPU: Tesla T4
Compute capability: (7, 5)
Compute dtype: torch.float16
BF16 hardware gate: disabled
Attention implementation: sdpa


In [3]:
# ================================================================
# 3. Frozen scientific constants
# ================================================================
MMMU_REPO = "MMMU/MMMU"
MMMU_REVISION = "98e6ac0cb9b7b2cd2c991b85a50762edc4aedc68"
MMMU_SPLITS = ("dev", "validation", "test")

MMMU_PRO_REPO = "MMMU/MMMU_Pro"
MMMU_PRO_CONFIG = "standard (10 options)"
MMMU_PRO_SPLIT = "test"
MMMU_PRO_REVISION = "563f3e84bb3b90893083a1f039cfa13077f2302b"
MMMU_PRO_EXPECTED_N = 1730

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
MODEL_REVISION = "89644892e4d85e24eaac8bacfd4f463576704203"

DOMAIN_TO_SUBJECTS = {
    "Art and Design": ["Art", "Art_Theory", "Design", "Music"],
    "Business": ["Accounting", "Economics", "Finance", "Manage", "Marketing"],
    "Science": ["Biology", "Chemistry", "Geography", "Math", "Physics"],
    "Health and Medicine": [
        "Basic_Medical_Science",
        "Clinical_Medicine",
        "Diagnostics_and_Laboratory_Medicine",
        "Pharmacy",
        "Public_Health",
    ],
    "Humanities and Social Science": [
        "History", "Literature", "Sociology", "Psychology",
    ],
    "Tech and Engineering": [
        "Agriculture",
        "Architecture_and_Engineering",
        "Computer_Science",
        "Electronics",
        "Energy_and_Power",
        "Materials",
        "Mechanical_Engineering",
    ],
}

SUBJECT_TO_DOMAIN = {
    subject: domain
    for domain, subjects in DOMAIN_TO_SUBJECTS.items()
    for subject in subjects
}

SUBJECTS = [
    subject
    for domain in DOMAIN_TO_SUBJECTS
    for subject in DOMAIN_TO_SUBJECTS[domain]
]

if len(SUBJECTS) != 30 or len(set(SUBJECTS)) != 30:
    raise RuntimeError("Expected exactly 30 unique subjects.")

SUBJECT_TO_ID = {subject: i for i, subject in enumerate(SUBJECTS)}
ID_TO_SUBJECT = {i: subject for subject, i in SUBJECT_TO_ID.items()}

WORKDIR = Path("/kaggle/working/mmmu_qwen3vl2b_domain_classifier")
WORKDIR.mkdir(parents=True, exist_ok=True)

AUDIT_DIR = WORKDIR / "audit"
CHECKPOINT_DIR = WORKDIR / "checkpoints"
FINAL_DIR = WORKDIR / "final_model"
for p in (AUDIT_DIR, CHECKPOINT_DIR, FINAL_DIR):
    p.mkdir(parents=True, exist_ok=True)

# Training configuration: fixed before held-out evaluation.
NUM_EPOCHS = 1
GRAD_ACCUM = 16
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_LR = 1.0e-4
HEAD_LR = 5.0e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0
CHECKPOINT_EVERY_OPT_STEPS = 50
CHECKPOINT_POLICY = "save_every_50_optimizer_steps"
RESUME = True

# Adaptive visual budget.
QWEN3VL_SPATIAL_COMPRESSION = 32
MIN_TOKENS_PER_IMAGE = 32
MAX_TOKENS_PER_IMAGE = 384
TOTAL_VISUAL_TOKENS_PER_SAMPLE = 640

# Extremely long samples are rejected rather than silently truncated.
MAX_SEQUENCE_TOKENS = 8192

print("Subjects:", len(SUBJECTS))
print("Domains:", len(DOMAIN_TO_SUBJECTS))

Subjects: 30
Domains: 6


## 4. Contamination-control policy

The following implementation reuses the project's conservative exact-overlap
definition: normalized IDs, normalized question stems, and canonical decoded
visual pixels are audited separately. Image hashing is based on decoded,
EXIF-corrected RGB content rather than compressed file bytes.

In [4]:
# ================================================================
# 4. Canonical normalization / image hashing helpers
# ================================================================
_TRANSLATION = str.maketrans({
    "‘": "'", "’": "'", "“": '"', "”": '"',
    "–": "-", "—": "-", "−": "-", " ": " ",
})

IMAGE_COLS = [f"image_{i}" for i in range(1, 8)]
IMAGE_MARKER_RE = re.compile(r"<image[\s_]*(\d+)\s*>", flags=re.IGNORECASE)

def normalize_id(x):
    if x is None:
        return ""
    return unicodedata.normalize("NFKC", str(x)).strip().casefold()

def normalize_question_exact(x):
    if x is None:
        return ""
    x = (
        unicodedata.normalize("NFKC", str(x))
        .translate(_TRANSLATION)
        .casefold()
    )
    return re.sub(r"\s+", " ", x).strip()

def image_to_pil(value):
    if value is None:
        return None
    if isinstance(value, Image.Image):
        return value.copy()
    if isinstance(value, dict):
        if value.get("bytes") is not None:
            with Image.open(io.BytesIO(value["bytes"])) as im:
                im.load()
                return im.copy()
        if value.get("path"):
            with Image.open(value["path"]) as im:
                im.load()
                return im.copy()
    if isinstance(value, (str, Path)):
        with Image.open(value) as im:
            im.load()
            return im.copy()
    raise TypeError(f"Unsupported image value: {type(value)!r}")

def canonical_rgb_hash(value):
    image = image_to_pil(value)
    if image is None:
        return None
    image = ImageOps.exif_transpose(image).convert("RGB")
    header = f"RGB|{image.width}|{image.height}|".encode("ascii")
    return hashlib.sha256(header + image.tobytes()).hexdigest()

def non_null_image_indices(row):
    out = []
    for i, col in enumerate(IMAGE_COLS, start=1):
        if col in row and row[col] is not None:
            out.append(i)
    return out

def parse_options(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple)):
        return [str(x) for x in value]
    text = str(value)
    try:
        parsed = py_ast.literal_eval(text)
        if isinstance(parsed, (list, tuple)):
            return [str(x) for x in parsed]
    except Exception:
        pass
    # Fallback: retain a single raw option string rather than dropping content.
    return [text]

def option_letter(i):
    return chr(ord("A") + i)

In [5]:
# ================================================================
# 5. Load the full MMMU-Pro contamination reference
# ================================================================
pro_ds = load_dataset(
    MMMU_PRO_REPO,
    MMMU_PRO_CONFIG,
    split=MMMU_PRO_SPLIT,
    revision=MMMU_PRO_REVISION,
)

if len(pro_ds) != MMMU_PRO_EXPECTED_N:
    raise RuntimeError(
        f"MMMU-Pro source-size mismatch: {len(pro_ds)} != {MMMU_PRO_EXPECTED_N}"
    )

pro_ids = set()
pro_questions = set()
pro_image_hashes = set()

for row in tqdm(pro_ds, desc="Indexing full MMMU-Pro contamination reference"):
    nid = normalize_id(row.get("id"))
    nq = normalize_question_exact(row.get("question"))
    if nid:
        pro_ids.add(nid)
    if nq:
        pro_questions.add(nq)

    for col in IMAGE_COLS:
        if col in row and row[col] is not None:
            digest = canonical_rgb_hash(row[col])
            if digest:
                pro_image_hashes.add(digest)

print("MMMU-Pro rows:", len(pro_ds))
print("Unique normalized IDs:", len(pro_ids))
print("Unique normalized questions:", len(pro_questions))
print("Unique canonical images:", len(pro_image_hashes))

README.md: 0.00B [00:00, ?B/s]

standard (10 options)/test-00000-of-0000(…):   0%|          | 0.00/346M [00:00<?, ?B/s]

standard (10 options)/test-00001-of-0000(…):   0%|          | 0.00/332M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1730 [00:00<?, ? examples/s]

Indexing full MMMU-Pro contamination reference:   0%|          | 0/1730 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


MMMU-Pro rows: 1730
Unique normalized IDs: 1730
Unique normalized questions: 1665
Unique canonical images: 1870


In [6]:
# ================================================================
# 6. Load ALL MMMU splits and construct an exact-clean training index
# ================================================================
MMMU_DATASETS = {}
clean_records = []
removed_records = []
source_counts = Counter()
clean_counts = Counter()
removal_counts = Counter()

for subject in SUBJECTS:
    for split in MMMU_SPLITS:
        ds = load_dataset(
            MMMU_REPO,
            subject,
            split=split,
            revision=MMMU_REVISION,
        )
        MMMU_DATASETS[(subject, split)] = ds

        for row_index in tqdm(
            range(len(ds)),
            desc=f"Cleaning {subject}/{split}",
            leave=False,
        ):
            row = ds[row_index]
            source_counts[(subject, split)] += 1

            nid = normalize_id(row.get("id"))
            nq = normalize_question_exact(row.get("question"))

            exact_id = bool(nid and nid in pro_ids)
            exact_question = bool(nq and nq in pro_questions)

            # Compute visual overlap only if the item was not already excluded
            # by deterministic text/ID provenance.
            exact_visual = False
            if not (exact_id or exact_question):
                for col in IMAGE_COLS:
                    if col in row and row[col] is not None:
                        digest = canonical_rgb_hash(row[col])
                        if digest and digest in pro_image_hashes:
                            exact_visual = True
                            break

            remove = exact_id or exact_question or exact_visual

            record = {
                "subject": subject,
                "domain": SUBJECT_TO_DOMAIN[subject],
                "split": split,
                "row_index": int(row_index),
                "id": str(row.get("id")),
            }

            if remove:
                removed_records.append({
                    **record,
                    "exact_id_overlap": exact_id,
                    "exact_question_overlap": exact_question,
                    "exact_visual_overlap": exact_visual,
                })
                removal_counts["any"] += 1
                removal_counts["id"] += int(exact_id)
                removal_counts["question"] += int(exact_question)
                removal_counts["visual_only"] += int(
                    exact_visual and not exact_id and not exact_question
                )
            else:
                clean_records.append(record)
                clean_counts[(subject, split)] += 1

source_total = sum(source_counts.values())
clean_total = len(clean_records)

audit_df = pd.DataFrame(removed_records)
clean_index_df = pd.DataFrame(clean_records)

audit_df.to_csv(AUDIT_DIR / "removed_mmmu_mmmupro_exact_overlap.csv", index=False)
clean_index_df.to_parquet(AUDIT_DIR / "clean_mmmu_training_index.parquet", index=False)

subject_stats = []
for subject in SUBJECTS:
    src = sum(source_counts[(subject, s)] for s in MMMU_SPLITS)
    kept = sum(clean_counts[(subject, s)] for s in MMMU_SPLITS)
    subject_stats.append({
        "subject": subject,
        "domain": SUBJECT_TO_DOMAIN[subject],
        "source_rows": src,
        "clean_rows": kept,
        "removed_rows": src - kept,
        "retention_pct": 100.0 * kept / src if src else float("nan"),
    })

subject_stats_df = pd.DataFrame(subject_stats)
subject_stats_df.to_csv(AUDIT_DIR / "cleaning_by_subject.csv", index=False)

summary = {
    "mmmu_revision": MMMU_REVISION,
    "mmmu_pro_revision": MMMU_PRO_REVISION,
    "mmmu_source_rows": source_total,
    "clean_training_rows": clean_total,
    "removed_any_exact_overlap": int(removal_counts["any"]),
    "removed_exact_id": int(removal_counts["id"]),
    "removed_exact_question": int(removal_counts["question"]),
    "removed_visual_only": int(removal_counts["visual_only"]),
    "policy": "remove if exact normalized ID OR exact normalized question OR exact canonical RGB visual overlap",
}
(AUDIT_DIR / "contamination_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)

print(json.dumps(summary, indent=2))
display(subject_stats_df)

README.md: 0.00B [00:00, ?B/s]

Art/dev-00000-of-00001.parquet:   0%|          | 0.00/6.25M [00:00<?, ?B/s]

Art/validation-00000-of-00001.parquet:   0%|          | 0.00/29.9M [00:00<?, ?B/s]

Art/test-00000-of-00001.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

...

Cleaning Mechanical_Engineering/dev:   0%|          | 0/5 [00:00<?, ?it/s]

Cleaning Mechanical_Engineering/validation:   0%|          | 0/30 [00:00<?, ?it/s]

Cleaning Mechanical_Engineering/test:   0%|          | 0/429 [00:00<?, ?it/s]

{
  "mmmu_revision": "98e6ac0cb9b7b2cd2c991b85a50762edc4aedc68",
  "mmmu_pro_revision": "563f3e84bb3b90893083a1f039cfa13077f2302b",
  "mmmu_source_rows": 11550,
  "clean_training_rows": 8426,
  "removed_any_exact_overlap": 3124,
  "removed_exact_id": 1730,
  "removed_exact_question": 2052,
  "removed_visual_only": 1054,
  "policy": "remove if exact normalized ID OR exact normalized question OR exact canonical RGB visual overlap"
}


,subject,domain,source_rows,clean_rows,removed_rows,retention_pct
0,Art,Art and Design,266,151,115,56.766917
1,Art_Theory,Art and Design,464,260,204,56.034483
2,Design,Art and Design,204,104,100,50.980392
3,Music,Art and Design,369,211,158,57.181572
4,Accounting,Business,415,295,120,71.084337
5,Economics,Business,302,158,144,52.317881
6,Finance,Business,390,241,149,61.794872
7,Manage,Business,280,209,71,74.642857
8,Marketing,Business,216,81,135,37.500000
9,Biology,Science,380,297,83,78.157895


In [7]:
# ================================================================
# 7. Held-out-500 reconstruction
# ================================================================
def find_unique_input_file(filename):
    roots = [Path("/kaggle/input"), Path("/kaggle/working")]
    matches = sorted({
        p.resolve()
        for root in roots
        if root.exists()
        for p in root.rglob(filename)
        if p.is_file()
    })
    if not matches:
        raise FileNotFoundError(f"Missing required Kaggle input: {filename}")
    return matches[0]

SELECTED_500_PATH = find_unique_input_file("selected_500_ids.txt")
selected_500_ids = [
    line.strip()
    for line in SELECTED_500_PATH.read_text(encoding="utf-8-sig").splitlines()
    if line.strip()
]

if len(selected_500_ids) != 500 or len(set(selected_500_ids)) != 500:
    raise RuntimeError("selected_500_ids.txt must contain exactly 500 unique IDs.")

pro_id_to_index = {
    str(row_id): i
    for i, row_id in enumerate(pro_ds["id"])
}

missing = [sid for sid in selected_500_ids if sid not in pro_id_to_index]
if missing:
    raise RuntimeError(f"Held-out IDs missing from pinned MMMU-Pro: {missing[:10]}")

heldout_indices = [pro_id_to_index[sid] for sid in selected_500_ids]
heldout_subjects = [str(pro_ds[i]["subject"]) for i in heldout_indices]

if set(heldout_subjects) != set(SUBJECTS):
    raise RuntimeError(
        "Held-out 500 does not cover the expected 30-subject taxonomy."
    )

heldout_subject_counts = Counter(heldout_subjects)

_count_values = list(heldout_subject_counts.values())
if not (
    _count_values.count(16) == 10
    and _count_values.count(17) == 20
    and sum(_count_values) == 500
):
    raise RuntimeError(
        "Held-out subject-count distribution does not match the frozen "
        "500-item cohort (10 subjects×16, 20 subjects×17)."
    )
print("Held-out 500 verified.")
print("Subject counts:", dict(sorted(heldout_subject_counts.items())))

Held-out 500 verified.
Subject counts: {'Accounting': 17, 'Agriculture': 16, 'Architecture_and_Engineering': 16, 'Art': 17, 'Art_Theory': 16, 'Basic_Medical_Science': 17, 'Biology': 17, 'Chemistry': 17, 'Clinical_Medicine': 16, 'Computer_Science': 17, 'Design': 17, 'Diagnostics_and_Laboratory_Medicine': 16, 'Economics': 16, 'Electronics': 16, 'Energy_and_Power': 16, 'Finance': 17, 'Geography': 17, 'History': 17, 'Literature': 17, 'Manage': 17, 'Marketing': 17, 'Materials': 17, 'Math': 17, 'Mechanical_Engineering': 17, 'Music': 17, 'Pharmacy': 17, 'Physics': 17, 'Psychology': 16, 'Public_Health': 16, 'Sociology': 17}


## 8. Multimodal serialization

The classifier input keeps semantic image placement. For example, an
image-valued option is represented conceptually as:

`Option C: [Image 3 | Option C] <actual image>`

If the same physical image is referenced again, the second occurrence becomes a
textual cross-reference to avoid duplicating visual tokens.

In [8]:
# ================================================================
# 8. Interleaved multimodal serializer
# ================================================================
SYSTEM_PROMPT = (
    "You are an academic-subject classifier. "
    "Do not solve the multiple-choice question. "
    "Use the question text, answer choices, and all visual evidence to encode "
    "which academic subject the problem belongs to."
)

FINAL_TASK_TEXT = (
    "\n\nClassification task: infer the single academic subject of this problem. "
    "The downstream classifier head maps this representation to one of the "
    "30 frozen MMMU subject labels."
)

def append_text(content, text):
    if not text:
        return
    if content and content[-1].get("type") == "text":
        content[-1]["text"] += text
    else:
        content.append({"type": "text", "text": text})

def collect_images(row):
    images = {}
    for idx, col in enumerate(IMAGE_COLS, start=1):
        if col in row and row[col] is not None:
            image = image_to_pil(row[col])
            image = ImageOps.exif_transpose(image).convert("RGB")
            images[idx] = image
    return images

def count_unique_available_images(row):
    return len(collect_images(row))

def visual_pixel_budget(n_unique_images):
    if n_unique_images <= 0:
        return None, None
    tokens_per_image = max(
        MIN_TOKENS_PER_IMAGE,
        min(
            MAX_TOKENS_PER_IMAGE,
            TOTAL_VISUAL_TOKENS_PER_SAMPLE // n_unique_images,
        ),
    )
    min_pixels = MIN_TOKENS_PER_IMAGE * (QWEN3VL_SPATIAL_COMPRESSION ** 2)
    max_pixels = tokens_per_image * (QWEN3VL_SPATIAL_COMPRESSION ** 2)
    return int(min_pixels), int(max_pixels)

def interleave_text_with_images(
    content,
    text,
    *,
    role,
    images_by_index,
    already_inserted,
    min_pixels,
    max_pixels,
):
    cursor = 0

    for match in IMAGE_MARKER_RE.finditer(str(text)):
        append_text(content, str(text)[cursor:match.start()])
        image_index = int(match.group(1))

        if image_index not in images_by_index:
            append_text(
                content,
                f"[Missing source Image {image_index} referenced in {role}]",
            )
        elif image_index in already_inserted:
            append_text(
                content,
                f"[Image {image_index} repeated; same visual shown earlier]",
            )
        else:
            append_text(
                content,
                f"\n[Image {image_index} | {role}]\n",
            )
            content.append({
                "type": "image",
                "image": images_by_index[image_index],
                "min_pixels": min_pixels,
                "max_pixels": max_pixels,
            })
            already_inserted.add(image_index)

        cursor = match.end()

    append_text(content, str(text)[cursor:])

def build_messages(row):
    images_by_index = collect_images(row)
    min_pixels, max_pixels = visual_pixel_budget(len(images_by_index))

    content = []
    inserted = set()

    append_text(content, "Question:\n")
    interleave_text_with_images(
        content,
        row.get("question", ""),
        role="Question",
        images_by_index=images_by_index,
        already_inserted=inserted,
        min_pixels=min_pixels,
        max_pixels=max_pixels,
    )

    options = parse_options(row.get("options"))
    if options:
        append_text(content, "\n\nAnswer choices:")
        for i, option in enumerate(options):
            letter = option_letter(i)
            append_text(content, f"\n{letter}. ")
            interleave_text_with_images(
                content,
                option,
                role=f"Option {letter}",
                images_by_index=images_by_index,
                already_inserted=inserted,
                min_pixels=min_pixels,
                max_pixels=max_pixels,
            )

    # Preserve source visuals that exist but were not referenced by a literal marker.
    for image_index in sorted(set(images_by_index) - inserted):
        append_text(
            content,
            f"\n\n[Additional source Image {image_index}]\n",
        )
        content.append({
            "type": "image",
            "image": images_by_index[image_index],
            "min_pixels": min_pixels,
            "max_pixels": max_pixels,
        })
        inserted.add(image_index)

    append_text(content, FINAL_TASK_TEXT)

    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": SYSTEM_PROMPT}],
        },
        {
            "role": "user",
            "content": content,
        },
    ]

# Sanity audit on a few multimodal rows.
checked = 0
for rec in clean_records:
    row = MMMU_DATASETS[(rec["subject"], rec["split"])][rec["row_index"]]
    if non_null_image_indices(row):
        messages = build_messages(row)
        image_items = sum(
            1
            for message in messages
            for item in message["content"]
            if item.get("type") == "image"
        )
        if image_items < 1:
            raise RuntimeError("Multimodal serialization lost all images.")
        checked += 1
    if checked >= 10:
        break

print("Multimodal serialization sanity checks:", checked, "PASS")

Multimodal serialization sanity checks: 10 PASS


In [9]:
# ================================================================
# 9. Load processor and configure Qwen3-VL visual budgets
# ================================================================
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

# Global defaults; individual image items also carry an adaptive budget.
processor.image_processor.size = {
    "shortest_edge": MIN_TOKENS_PER_IMAGE * (QWEN3VL_SPATIAL_COMPRESSION ** 2),
    "longest_edge": MAX_TOKENS_PER_IMAGE * (QWEN3VL_SPATIAL_COMPRESSION ** 2),
}

processor.tokenizer.padding_side = "right"

def encode_row(row):
    messages = build_messages(row)

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True,
        return_tensors="pt",
    )

    seq_len = int(inputs["input_ids"].shape[-1])
    if seq_len > MAX_SEQUENCE_TOKENS:
        raise RuntimeError(
            f"Encoded sequence length {seq_len} exceeds "
            f"MAX_SEQUENCE_TOKENS={MAX_SEQUENCE_TOKENS}. "
            "This run forbids silent truncation."
        )

    return inputs

# Processor smoke test: one text-only and one multimodal item if available.
smoke_done = {"text": False, "image": False}

for rec in clean_records:
    row = MMMU_DATASETS[(rec["subject"], rec["split"])][rec["row_index"]]
    has_image = bool(non_null_image_indices(row))
    key = "image" if has_image else "text"
    if smoke_done[key]:
        continue

    test_inputs = encode_row(row)
    print(
        f"{key} smoke test:",
        rec["id"],
        "tokens=", int(test_inputs["input_ids"].shape[-1]),
        "keys=", sorted(test_inputs.keys()),
    )
    smoke_done[key] = True

    if all(smoke_done.values()):
        break

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

image smoke test: dev_Art_1 tokens= 515 keys= ['attention_mask', 'image_grid_thw', 'input_ids', 'mm_token_type_ids', 'pixel_values']


## 10. Model architecture

`Qwen3-VL-2B-Instruct` is loaded in NF4 4-bit form. The vision encoder remains
frozen. LoRA adapters are attached only to the language decoder's attention and
MLP projections:

- `q_proj`, `k_proj`, `v_proj`, `o_proj`
- `gate_proj`, `up_proj`, `down_proj`

A trainable `LayerNorm → Dropout → Linear(30)` head consumes the final
non-padding multimodal hidden state.

This is a discriminative classifier: it does not generate a subject string.

In [10]:
# ================================================================
# 10. QLoRA backbone + 30-way classification head
# ================================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=COMPUTE_DTYPE,
    attn_implementation=ATTN_IMPLEMENTATION,
)

base_model.config.use_cache = False

base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True,
)

LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

# Resume adapter if present; otherwise inject a new adapter.
def checkpoint_step(path):
    m = re.fullmatch(r"checkpoint_step_(\d+)", path.name)
    return int(m.group(1)) if m else -1

def latest_checkpoint():
    candidates = [
        p for p in CHECKPOINT_DIR.glob("checkpoint_step_*")
        if p.is_dir() and (p / "adapter").is_dir() and (p / "trainer_state.pt").is_file()
    ]
    if not candidates:
        return None
    return max(candidates, key=checkpoint_step)

RESUME_CHECKPOINT = latest_checkpoint() if RESUME else None

if RESUME_CHECKPOINT is not None:
    backbone = PeftModel.from_pretrained(
        base_model,
        RESUME_CHECKPOINT / "adapter",
        is_trainable=True,
    )
    print("Resuming adapter from:", RESUME_CHECKPOINT)
else:
    backbone = get_peft_model(base_model, lora_config)
    print("Created new LoRA adapter.")

# Verify LoRA did not enter the vision tower.
trainable_lora_names = [
    name
    for name, param in backbone.named_parameters()
    if param.requires_grad and "lora_" in name
]

if not trainable_lora_names:
    raise RuntimeError("No trainable LoRA parameters were created.")

vision_lora = [name for name in trainable_lora_names if ".visual." in name]
if vision_lora:
    raise RuntimeError(
        "LoRA unexpectedly entered the vision tower: "
        + repr(vision_lora[:10])
    )

# Resolve hidden size from the text configuration.
base_config = backbone.get_base_model().config
hidden_size = int(base_config.text_config.hidden_size)

class Qwen3VLSubjectClassifier(nn.Module):
    def __init__(self, peft_backbone, hidden_size, num_labels):
        super().__init__()
        self.backbone = peft_backbone
        self.norm = nn.LayerNorm(hidden_size, dtype=torch.float32)
        self.dropout = nn.Dropout(0.10)
        self.classifier = nn.Linear(hidden_size, num_labels, dtype=torch.float32)

    def _core_multimodal_model(self):
        base = self.backbone.get_base_model()
        if not hasattr(base, "model"):
            raise RuntimeError(
                f"Expected Qwen3-VL conditional model with .model, got {type(base)!r}"
            )
        return base.model

    def forward(self, model_inputs):
        core = self._core_multimodal_model()

        # Only arguments accepted by the multimodal base model are forwarded.
        allowed = {
            "input_ids",
            "attention_mask",
            "position_ids",
            "pixel_values",
            "pixel_values_videos",
            "image_grid_thw",
            "video_grid_thw",
            "mm_token_type_ids",
        }
        forwarded = {
            k: v
            for k, v in model_inputs.items()
            if k in allowed
        }

        outputs = core(
            **forwarded,
            use_cache=False,
            return_dict=True,
        )

        hidden = outputs.last_hidden_state
        attention_mask = model_inputs["attention_mask"]

        last_index = attention_mask.long().sum(dim=1) - 1
        batch_index = torch.arange(hidden.shape[0], device=hidden.device)
        pooled = hidden[batch_index, last_index]

        pooled = self.norm(pooled.float())
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits

model = Qwen3VLSubjectClassifier(
    backbone,
    hidden_size=hidden_size,
    num_labels=len(SUBJECTS),
).cuda()

# Restore classifier head if resuming.
if RESUME_CHECKPOINT is not None:
    head_state = torch.load(
        RESUME_CHECKPOINT / "classifier_head.pt",
        map_location="cpu",
    )
    model.norm.load_state_dict(head_state["norm"])
    model.classifier.load_state_dict(head_state["classifier"])

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())

print(f"Trainable parameters: {n_trainable:,}")
print(f"Visible parameter count: {n_total:,}")
print(f"Trainable fraction: {100*n_trainable/n_total:.4f}%")
print("Trainable LoRA tensors:", len(trainable_lora_names))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Created new LoRA adapter.
Trainable parameters: 17,498,142
Visible parameter count: 1,239,060,510
Trainable fraction: 1.4122%
Trainable LoRA tensors: 392


In [11]:
# ================================================================
# 11. Training weights and deterministic order
# ================================================================
subject_frequency = Counter(rec["subject"] for rec in clean_records)

# Inverse-sqrt weighting: reduces large-subject dominance without making
# minority classes excessively noisy.
median_count = float(np.median(list(subject_frequency.values())))
class_weights_np = np.array([
    math.sqrt(median_count / subject_frequency[subject])
    for subject in SUBJECTS
], dtype=np.float32)

class_weights_np = np.clip(class_weights_np, 0.5, 2.0)
class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device="cuda")

class_weight_df = pd.DataFrame({
    "subject": SUBJECTS,
    "domain": [SUBJECT_TO_DOMAIN[s] for s in SUBJECTS],
    "clean_train_rows": [subject_frequency[s] for s in SUBJECTS],
    "loss_weight": class_weights_np,
})
class_weight_df.to_csv(AUDIT_DIR / "subject_class_weights.csv", index=False)
display(class_weight_df)

num_micro_steps = len(clean_records) * NUM_EPOCHS
num_optimizer_steps = math.ceil(num_micro_steps / GRAD_ACCUM)
warmup_steps = max(1, int(num_optimizer_steps * WARMUP_RATIO))

print("Clean training samples:", len(clean_records))
print("Epochs:", NUM_EPOCHS)
print("Gradient accumulation:", GRAD_ACCUM)
print("Expected optimizer steps:", num_optimizer_steps)
print("Warmup steps:", warmup_steps)

,subject,domain,clean_train_rows,loss_weight
0,Art,Art and Design,151,1.315974
1,Art_Theory,Art and Design,260,1.002880
2,Design,Art and Design,104,1.585693
3,Music,Art and Design,211,1.113255
4,Accounting,Business,295,0.941510
5,Economics,Business,158,1.286493
6,Finance,Business,241,1.041663
7,Manage,Business,209,1.118569
8,Marketing,Business,81,1.796774
9,Biology,Science,297,0.938334


Clean training samples: 8426
Epochs: 1
Gradient accumulation: 16
Expected optimizer steps: 527
Warmup steps: 15


In [12]:
# ================================================================
# 12. Optimizer, scheduler, checkpoint / resume
# ================================================================
lora_params = []
head_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if name.startswith("norm.") or name.startswith("classifier."):
        head_params.append(param)
    else:
        lora_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {
            "params": lora_params,
            "lr": LORA_LR,
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": head_params,
            "lr": HEAD_LR,
            "weight_decay": WEIGHT_DECAY,
        },
    ]
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=num_optimizer_steps,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(COMPUTE_DTYPE == torch.float16),
)

state = {
    "epoch": 0,
    "sample_position": 0,
    "global_optimizer_step": 0,
    "running_loss_sum": 0.0,
    "running_examples": 0,
}

def move_optimizer_state_to_cuda(optimizer):
    for opt_state in optimizer.state.values():
        for key, value in list(opt_state.items()):
            if torch.is_tensor(value):
                opt_state[key] = value.cuda()

if RESUME_CHECKPOINT is not None:
    saved = torch.load(
        RESUME_CHECKPOINT / "trainer_state.pt",
        map_location="cpu",
    )
    optimizer.load_state_dict(saved["optimizer"])
    scheduler.load_state_dict(saved["scheduler"])
    if saved.get("scaler") is not None:
        scaler.load_state_dict(saved["scaler"])
    state.update(saved["state"])
    move_optimizer_state_to_cuda(optimizer)
    print("Training state restored:", state)

def save_checkpoint(global_step, state):
    ckpt = CHECKPOINT_DIR / f"checkpoint_step_{global_step:06d}"
    ckpt.mkdir(parents=True, exist_ok=True)

    model.backbone.save_pretrained(ckpt / "adapter")

    torch.save({
        "norm": model.norm.state_dict(),
        "classifier": model.classifier.state_dict(),
        "subject_to_id": SUBJECT_TO_ID,
        "subject_to_domain": SUBJECT_TO_DOMAIN,
        "hidden_size": hidden_size,
    }, ckpt / "classifier_head.pt")

    torch.save({
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict() if scaler.is_enabled() else None,
        "state": dict(state),
    }, ckpt / "trainer_state.pt")

    (ckpt / "config.json").write_text(
        json.dumps({
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "mmmu_revision": MMMU_REVISION,
            "mmmu_pro_revision": MMMU_PRO_REVISION,
            "num_subjects": len(SUBJECTS),
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT,
            "lora_lr": LORA_LR,
            "head_lr": HEAD_LR,
            "grad_accum": GRAD_ACCUM,
            "compute_dtype": str(COMPUTE_DTYPE),
        }, indent=2),
        encoding="utf-8",
    )

    print("Checkpoint saved:", ckpt)

## Pre-training runtime canary

Before the full one-epoch run, one clean MMMU example is passed through the
**same serializer, Qwen3-VL backbone, LoRA adapter, classification head, loss,
autocast, and backward path** used by training.

This is a fail-fast engineering check only: it performs **no optimizer step**
and restores RNG state afterward, so it does not change the scientific training
protocol. Its purpose is to catch processor/API incompatibility, unsupported
precision, CUDA OOM, or backward-path problems before hours of training.

In [13]:
# ================================================================
# 12A. Fail-fast forward/backward canary (NO optimizer step)
# ================================================================
def _row_image_count(row):
    count = 0
    for i in range(1, 64):
        key = f"image_{i}"
        if key not in row:
            break
        if row.get(key) is not None:
            count += 1
    return count


# Prefer a multimodal example so the canary exercises the vision path.
canary_record = None
canary_row = None

for candidate in clean_records:
    row = MMMU_DATASETS[
        (candidate["subject"], candidate["split"])
    ][candidate["row_index"]]

    if _row_image_count(row) >= 1:
        canary_record = candidate
        canary_row = row
        break

if canary_record is None:
    raise RuntimeError(
        "Could not find a multimodal clean MMMU row for the runtime canary."
    )

# Preserve deterministic training RNG state.
_py_rng = random.getstate()
_np_rng = np.random.get_state()
_torch_rng = torch.get_rng_state()
_cuda_rng = torch.cuda.get_rng_state_all()

try:
    model.train()
    optimizer.zero_grad(set_to_none=True)

    canary_inputs = encode_row(canary_row)
    canary_inputs = {
        k: (
            v.cuda(non_blocking=True)
            if torch.is_tensor(v)
            else v
        )
        for k, v in canary_inputs.items()
    }

    canary_target = torch.tensor(
        [SUBJECT_TO_ID[canary_record["subject"]]],
        dtype=torch.long,
        device="cuda",
    )

    with torch.autocast(
        device_type="cuda",
        dtype=COMPUTE_DTYPE,
        enabled=True,
    ):
        canary_logits = model(canary_inputs)
        canary_loss = F.cross_entropy(
            canary_logits,
            canary_target,
            weight=class_weights,
        )

    if canary_logits.shape != (1, len(SUBJECTS)):
        raise RuntimeError(
            "Canary logits shape mismatch: "
            f"{tuple(canary_logits.shape)}"
        )

    if not torch.isfinite(canary_loss).item():
        raise RuntimeError(
            f"Canary produced non-finite loss: {canary_loss.item()}"
        )

    # Real backward path; deliberately no optimizer.step().
    canary_loss.backward()

    trainable_with_grad = sum(
        1
        for p in model.parameters()
        if p.requires_grad and p.grad is not None
    )

    if trainable_with_grad == 0:
        raise RuntimeError(
            "Canary backward produced no trainable gradients."
        )

    print(
        "PRE-TRAINING CANARY: PASS | "
        f"id={canary_record['id']} | "
        f"subject={canary_record['subject']} | "
        f"images={_row_image_count(canary_row)} | "
        f"seq_len={int(canary_inputs['input_ids'].shape[-1])} | "
        f"logits={tuple(canary_logits.shape)} | "
        f"loss={float(canary_loss.detach().cpu()):.6f} | "
        f"dtype={COMPUTE_DTYPE} | "
        f"trainable_grad_tensors={trainable_with_grad}"
    )

finally:
    optimizer.zero_grad(set_to_none=True)

    for name in (
        "canary_inputs",
        "canary_logits",
        "canary_loss",
        "canary_target",
    ):
        if name in globals():
            del globals()[name]

    gc.collect()
    torch.cuda.empty_cache()

    random.setstate(_py_rng)
    np.random.set_state(_np_rng)
    torch.set_rng_state(_torch_rng)
    torch.cuda.set_rng_state_all(_cuda_rng)

print("Training protocol unchanged: canary performed zero optimizer steps.")

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


PRE-TRAINING CANARY: PASS | id=dev_Art_1 | subject=Art | images=1 | seq_len=515 | logits=(1, 30) | loss=3.777344 | dtype=torch.float16 | trainable_grad_tensors=396
Training protocol unchanged: canary performed zero optimizer steps.


In [14]:
# ================================================================
# 13. Fixed one-epoch-all-clean-data training loop
# ================================================================
def epoch_order(epoch_index):
    indices = list(range(len(clean_records)))
    rng = random.Random(SEED + 1000 * epoch_index)
    rng.shuffle(indices)
    return indices

def get_clean_row(record):
    return MMMU_DATASETS[
        (record["subject"], record["split"])
    ][record["row_index"]]

model.train()
optimizer.zero_grad(set_to_none=True)

train_log_path = WORKDIR / "training_log.jsonl"

for epoch in range(state["epoch"], NUM_EPOCHS):
    order = epoch_order(epoch)
    start_pos = state["sample_position"] if epoch == state["epoch"] else 0

    pbar = tqdm(
        range(start_pos, len(order)),
        desc=f"Training epoch {epoch+1}/{NUM_EPOCHS}",
    )

    accum_counter = 0

    for pos in pbar:
        record = clean_records[order[pos]]
        row = get_clean_row(record)
        inputs = encode_row(row)

        # Batch size is intentionally one because visual counts and resolutions vary.
        inputs = {
            k: v.cuda(non_blocking=True) if torch.is_tensor(v) else v
            for k, v in inputs.items()
        }

        target = torch.tensor(
            [SUBJECT_TO_ID[record["subject"]]],
            dtype=torch.long,
            device="cuda",
        )

        autocast_ctx = torch.autocast(
            device_type="cuda",
            dtype=COMPUTE_DTYPE,
            enabled=True,
        )

        with autocast_ctx:
            logits = model(inputs)
            loss = F.cross_entropy(
                logits,
                target,
                weight=class_weights,
            )

        raw_loss = float(loss.detach().cpu())
        scaled_loss = loss / GRAD_ACCUM

        if scaler.is_enabled():
            scaler.scale(scaled_loss).backward()
        else:
            scaled_loss.backward()

        accum_counter += 1
        state["running_loss_sum"] += raw_loss
        state["running_examples"] += 1
        state["epoch"] = epoch
        state["sample_position"] = pos + 1

        should_step = (
            accum_counter >= GRAD_ACCUM
            or pos == len(order) - 1
        )

        if should_step:
            if scaler.is_enabled():
                scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                MAX_GRAD_NORM,
            )

            if scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            accum_counter = 0
            state["global_optimizer_step"] += 1

            mean_loss = (
                state["running_loss_sum"] / state["running_examples"]
            )

            log_record = {
                "time_utc": pd.Timestamp.utcnow().isoformat(),
                "epoch": epoch,
                "sample_position": pos + 1,
                "global_optimizer_step": state["global_optimizer_step"],
                "mean_loss": mean_loss,
                "lr_lora": optimizer.param_groups[0]["lr"],
                "lr_head": optimizer.param_groups[1]["lr"],
                "gpu": GPU_NAME,
            }

            with train_log_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(log_record) + "\n")

            pbar.set_postfix({
                "loss": f"{mean_loss:.4f}",
                "opt_step": state["global_optimizer_step"],
            })

            if (
                state["global_optimizer_step"] % CHECKPOINT_EVERY_OPT_STEPS == 0
            ):
                save_checkpoint(
                    state["global_optimizer_step"],
                    state,
                )

        # Avoid retaining processor / PIL tensors.
        del inputs, logits, loss, scaled_loss
        if pos % 100 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    # Epoch completed.
    state["epoch"] = epoch + 1
    state["sample_position"] = 0
    save_checkpoint(state["global_optimizer_step"], state)

print("Training complete.")

Training epoch 1/1:   0%|          | 0/8426 [00:00<?, ?it/s]

/tmp/ipykernel_23/3481746151.py:96: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Checkpoint saved: /kaggle/working/mmmu_qwen3vl2b_domain_classifier/checkpoints/checkpoint_step_000050


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Checkpoint saved: /kaggle/working/mmmu_qwen3vl2b_domain_classifier/checkpoints/checkpoint_step_000100
Checkpoint saved: /kaggle/working/mmmu_qwen3vl2b_domain_classifier/checkpoints/checkpoint_step_000150
Checkpoint saved: /kaggle/working/mmmu_qwen3vl2b_domain_classifier/checkpoints/checkpoint_step_000200
Checkpoint saved: /kaggle/working/mmmu_qwen3vl2b_domain_classifier/checkpoints/checkpoint_step_000250
Checkpoint saved: /kaggle/working/mmmu_qwen3vl2b_domain_classifier/checkpoints/checkpoint_step_000300
Checkpoint saved: /kaggle/working/mmmu_qwen3vl2b_domain_classifier/checkpoints/checkpoint_step_000350
Checkpoint saved: /kaggle/working/mmmu_qwen3vl2b_domain_classifier/checkpoints/checkpoint_step_000400
Checkpoint saved: /kaggle/working/mmmu_qwen3vl2b_domain_classifier/checkpoints/checkpoint_step_000450
Checkpoint saved: /kaggle/working/mmmu_qwen3vl2b_domain_classifier/checkpoints/checkpoint_step_000500
Checkpoint saved: /kaggle/working/mmmu_qwen3vl2b_domain_classifier/checkpoints/che

In [15]:
# ================================================================
# 14. Save final adapter + classification head
# ================================================================
FINAL_DIR.mkdir(parents=True, exist_ok=True)

model.backbone.save_pretrained(FINAL_DIR / "adapter")
processor.save_pretrained(FINAL_DIR / "processor")

torch.save({
    "norm": model.norm.state_dict(),
    "classifier": model.classifier.state_dict(),
    "subject_to_id": SUBJECT_TO_ID,
    "id_to_subject": ID_TO_SUBJECT,
    "subject_to_domain": SUBJECT_TO_DOMAIN,
    "hidden_size": hidden_size,
}, FINAL_DIR / "classifier_head.pt")

ROUTER_CLASSIFIER_ARTIFACT_VERSION = "qwen3vl2b-subject-domain-router-v1"

final_manifest = {
    "router_classifier_artifact_version": ROUTER_CLASSIFIER_ARTIFACT_VERSION,
    "router_contract": "predict(question, images, options) -> {subject, domain, metadata}",
    "architecture": "Qwen3-VL-2B-Instruct NF4 QLoRA + 30-way discriminative head",
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "vision_tower_trainable": False,
    "lora_target_modules": LORA_TARGET_MODULES,
    "training_rows": len(clean_records),
    "training_epochs": NUM_EPOCHS,
    "checkpoint_every_optimizer_steps": CHECKPOINT_EVERY_OPT_STEPS,
    "checkpoint_policy": CHECKPOINT_POLICY,
    "mmmu_revision": MMMU_REVISION,
    "mmmu_pro_revision": MMMU_PRO_REVISION,
    "contamination_policy": (
        "exclude exact normalized ID OR exact normalized question "
        "OR exact canonical RGB visual overlap against all 1,730 MMMU-Pro rows"
    ),
    "heldout500_used_during_training": False,
    "subject_labels": SUBJECTS,
    "subject_to_domain": SUBJECT_TO_DOMAIN,
    "visual_budget": {
        "min_tokens_per_image": MIN_TOKENS_PER_IMAGE,
        "max_tokens_per_image": MAX_TOKENS_PER_IMAGE,
        "total_tokens_per_sample": TOTAL_VISUAL_TOKENS_PER_SAMPLE,
    },
}

(FINAL_DIR / "training_manifest.json").write_text(
    json.dumps(final_manifest, indent=2),
    encoding="utf-8",
)

print("Final model artifacts:", FINAL_DIR)
# ----------------------------------------------------------------
# Export router-facing inference adapter
# ----------------------------------------------------------------
ROUTER_CLASSIFIER_SOURCE = '# Auto-generated by the MMMU Qwen3-VL training notebook.\n# Contract:\n# predict(question=..., images=..., options=...) ->\n# {"subject": "...", "domain": "...", ...metadata...}\n\nfrom __future__ import annotations\n\nimport ast as py_ast\nimport io\nimport json\nimport math\nimport re\nfrom pathlib import Path\nfrom typing import Any, Iterable, Optional\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom PIL import Image, ImageOps\nfrom peft import PeftModel\nfrom transformers import (\n    AutoProcessor,\n    AutoModelForImageTextToText,\n    BitsAndBytesConfig,\n)\n\nROUTER_CLASSIFIER_ARTIFACT_VERSION = "qwen3vl2b-subject-domain-router-v1"\n\nQWEN3VL_SPATIAL_COMPRESSION = 32\nMIN_TOKENS_PER_IMAGE = 32\nMAX_TOKENS_PER_IMAGE = 384\nTOTAL_VISUAL_TOKENS_PER_SAMPLE = 640\nMAX_SEQUENCE_TOKENS = 8192\n\nIMAGE_MARKER_RE = re.compile(\n    r"<image[\\s_]*(\\d+)\\s*>",\n    flags=re.IGNORECASE,\n)\n\nSYSTEM_PROMPT = (\n    "You are an academic-subject classifier. "\n    "Do not solve the multiple-choice question. "\n    "Use the question text, answer choices, and all visual evidence to encode "\n    "which academic subject the problem belongs to."\n)\n\nFINAL_TASK_TEXT = (\n    "\\n\\nClassification task: infer the single academic subject of this problem. "\n    "The downstream classifier head maps this representation to one of the "\n    "30 frozen MMMU subject labels."\n)\n\n\ndef _append_text(content: list[dict], text: str) -> None:\n    if not text:\n        return\n    if content and content[-1].get("type") == "text":\n        content[-1]["text"] += text\n    else:\n        content.append({"type": "text", "text": text})\n\n\ndef _image_to_pil(value: Any) -> Image.Image:\n    if isinstance(value, Image.Image):\n        image = value.copy()\n        image.load()\n        return image\n\n    if isinstance(value, memoryview):\n        value = value.tobytes()\n\n    if isinstance(value, (bytes, bytearray)):\n        with Image.open(io.BytesIO(bytes(value))) as im:\n            im.load()\n            return im.copy()\n\n    if isinstance(value, dict):\n        raw = value.get("bytes")\n        if isinstance(raw, memoryview):\n            raw = raw.tobytes()\n        if isinstance(raw, (bytes, bytearray)):\n            with Image.open(io.BytesIO(bytes(raw))) as im:\n                im.load()\n                return im.copy()\n        if value.get("path"):\n            value = value["path"]\n\n    if isinstance(value, (str, Path)):\n        with Image.open(value) as im:\n            im.load()\n            return im.copy()\n\n    raise TypeError(\n        "Classifier image must be PIL.Image, bytes, a local path, "\n        "or a dict containing bytes/path."\n    )\n\n\ndef _choice_letter(index: int) -> str:\n    return chr(ord("A") + int(index))\n\n\ndef _visual_pixel_budget(n_unique_images: int):\n    if n_unique_images <= 0:\n        return None, None\n\n    tokens_per_image = max(\n        MIN_TOKENS_PER_IMAGE,\n        min(\n            MAX_TOKENS_PER_IMAGE,\n            TOTAL_VISUAL_TOKENS_PER_SAMPLE // n_unique_images,\n        ),\n    )\n\n    min_pixels = (\n        MIN_TOKENS_PER_IMAGE\n        * (QWEN3VL_SPATIAL_COMPRESSION ** 2)\n    )\n    max_pixels = (\n        tokens_per_image\n        * (QWEN3VL_SPATIAL_COMPRESSION ** 2)\n    )\n\n    return int(min_pixels), int(max_pixels)\n\n\ndef _interleave_text_with_images(\n    content: list[dict],\n    text: str,\n    *,\n    role: str,\n    images_by_index: dict[int, Image.Image],\n    already_inserted: set[int],\n    min_pixels: Optional[int],\n    max_pixels: Optional[int],\n) -> None:\n    cursor = 0\n    text = str(text)\n\n    for match in IMAGE_MARKER_RE.finditer(text):\n        _append_text(content, text[cursor:match.start()])\n        image_index = int(match.group(1))\n\n        if image_index not in images_by_index:\n            _append_text(\n                content,\n                f"[Missing source Image {image_index} referenced in {role}]",\n            )\n        elif image_index in already_inserted:\n            _append_text(\n                content,\n                f"[Image {image_index} repeated; same visual shown earlier]",\n            )\n        else:\n            _append_text(\n                content,\n                f"\\n[Image {image_index} | {role}]\\n",\n            )\n            content.append({\n                "type": "image",\n                "image": images_by_index[image_index],\n                "min_pixels": min_pixels,\n                "max_pixels": max_pixels,\n            })\n            already_inserted.add(image_index)\n\n        cursor = match.end()\n\n    _append_text(content, text[cursor:])\n\n\ndef _build_messages(\n    question: str,\n    images: Iterable[Any],\n    options: Iterable[str],\n) -> list[dict]:\n    image_list = [\n        ImageOps.exif_transpose(\n            _image_to_pil(image)\n        ).convert("RGB")\n        for image in images\n    ]\n\n    images_by_index = {\n        i + 1: image\n        for i, image in enumerate(image_list)\n    }\n\n    min_pixels, max_pixels = _visual_pixel_budget(\n        len(images_by_index)\n    )\n\n    content: list[dict] = []\n    inserted: set[int] = set()\n\n    _append_text(content, "Question:\\n")\n    _interleave_text_with_images(\n        content,\n        str(question),\n        role="Question",\n        images_by_index=images_by_index,\n        already_inserted=inserted,\n        min_pixels=min_pixels,\n        max_pixels=max_pixels,\n    )\n\n    option_list = [str(x) for x in options]\n    if option_list:\n        _append_text(content, "\\n\\nAnswer choices:")\n\n        for i, option in enumerate(option_list):\n            letter = _choice_letter(i)\n            _append_text(content, f"\\n{letter}. ")\n            _interleave_text_with_images(\n                content,\n                option,\n                role=f"Option {letter}",\n                images_by_index=images_by_index,\n                already_inserted=inserted,\n                min_pixels=min_pixels,\n                max_pixels=max_pixels,\n            )\n\n    # Preserve any physical image that exists but is not referenced\n    # by an explicit <image n> marker.\n    for image_index in sorted(\n        set(images_by_index) - inserted\n    ):\n        _append_text(\n            content,\n            f"\\n\\n[Additional source Image {image_index}]\\n",\n        )\n        content.append({\n            "type": "image",\n            "image": images_by_index[image_index],\n            "min_pixels": min_pixels,\n            "max_pixels": max_pixels,\n        })\n        inserted.add(image_index)\n\n    _append_text(content, FINAL_TASK_TEXT)\n\n    return [\n        {\n            "role": "system",\n            "content": [\n                {\n                    "type": "text",\n                    "text": SYSTEM_PROMPT,\n                }\n            ],\n        },\n        {\n            "role": "user",\n            "content": content,\n        },\n    ]\n\n\nclass _Qwen3VLSubjectClassifier(nn.Module):\n    def __init__(\n        self,\n        peft_backbone,\n        hidden_size: int,\n        num_labels: int,\n    ):\n        super().__init__()\n        self.backbone = peft_backbone\n        self.norm = nn.LayerNorm(\n            hidden_size,\n            dtype=torch.float32,\n        )\n        self.dropout = nn.Dropout(0.10)\n        self.classifier = nn.Linear(\n            hidden_size,\n            num_labels,\n            dtype=torch.float32,\n        )\n\n    def _core_multimodal_model(self):\n        base = self.backbone.get_base_model()\n\n        if not hasattr(base, "model"):\n            raise RuntimeError(\n                "Expected Qwen3-VL conditional model with .model; "\n                f"observed {type(base)!r}."\n            )\n\n        return base.model\n\n    def forward(self, model_inputs: dict):\n        core = self._core_multimodal_model()\n\n        allowed = {\n            "input_ids",\n            "attention_mask",\n            "position_ids",\n            "pixel_values",\n            "pixel_values_videos",\n            "image_grid_thw",\n            "video_grid_thw",\n            "mm_token_type_ids",\n        }\n\n        forwarded = {\n            key: value\n            for key, value in model_inputs.items()\n            if key in allowed\n        }\n\n        outputs = core(\n            **forwarded,\n            use_cache=False,\n            return_dict=True,\n        )\n\n        hidden = outputs.last_hidden_state\n        attention_mask = model_inputs["attention_mask"]\n\n        last_index = (\n            attention_mask.long().sum(dim=1) - 1\n        )\n\n        batch_index = torch.arange(\n            hidden.shape[0],\n            device=hidden.device,\n        )\n\n        pooled = hidden[\n            batch_index,\n            last_index,\n        ]\n\n        pooled = self.norm(\n            pooled.float()\n        )\n        pooled = self.dropout(\n            pooled\n        )\n\n        return self.classifier(\n            pooled\n        )\n\n\nclass Qwen3VLRouterClassifier:\n    """\n    Router-facing adapter for the trained 30-subject Qwen3-VL model.\n\n    Public contract:\n        predict(question=..., images=..., options=...) -> dict\n    """\n\n    def __init__(\n        self,\n        model_dir: str | Path,\n        *,\n        device: Optional[str] = None,\n    ):\n        self.model_dir = Path(model_dir).resolve()\n\n        required = [\n            self.model_dir / "training_manifest.json",\n            self.model_dir / "classifier_head.pt",\n            self.model_dir / "adapter",\n            self.model_dir / "processor",\n        ]\n\n        missing = [\n            str(path)\n            for path in required\n            if not path.exists()\n        ]\n\n        if missing:\n            raise FileNotFoundError(\n                "Incomplete trained-classifier artifact: "\n                + repr(missing)\n            )\n\n        self.manifest = json.loads(\n            (\n                self.model_dir\n                / "training_manifest.json"\n            ).read_text(\n                encoding="utf-8"\n            )\n        )\n\n        artifact_version = self.manifest.get(\n            "router_classifier_artifact_version"\n        )\n\n        if (\n            artifact_version\n            != ROUTER_CLASSIFIER_ARTIFACT_VERSION\n        ):\n            raise RuntimeError(\n                "Router classifier artifact-version mismatch: "\n                f"{artifact_version!r} != "\n                f"{ROUTER_CLASSIFIER_ARTIFACT_VERSION!r}"\n            )\n\n        if not torch.cuda.is_available():\n            raise RuntimeError(\n                "Qwen3-VL trained classifier requires a CUDA GPU."\n            )\n\n        self.device = torch.device(\n            device or "cuda:0"\n        )\n\n        gpu_major = torch.cuda.get_device_capability(\n            self.device\n        )[0]\n\n        self.compute_dtype = (\n            torch.bfloat16\n            if (\n                gpu_major >= 8\n                and torch.cuda.is_bf16_supported()\n            )\n            else torch.float16\n        )\n\n        self.attn_implementation = (\n            "sdpa"\n            if gpu_major >= 7\n            else "eager"\n        )\n\n        head_state = torch.load(\n            self.model_dir\n            / "classifier_head.pt",\n            map_location="cpu",\n        )\n\n        self.subject_to_id = {\n            str(key): int(value)\n            for key, value\n            in head_state[\n                "subject_to_id"\n            ].items()\n        }\n\n        self.id_to_subject = {\n            int(key): str(value)\n            for key, value\n            in head_state[\n                "id_to_subject"\n            ].items()\n        }\n\n        self.subject_to_domain = {\n            str(key): str(value)\n            for key, value\n            in head_state[\n                "subject_to_domain"\n            ].items()\n        }\n\n        if len(self.subject_to_id) != 30:\n            raise RuntimeError(\n                "Expected exactly 30 subject labels."\n            )\n\n        if set(self.subject_to_id) != set(\n            self.subject_to_domain\n        ):\n            raise RuntimeError(\n                "Subject/domain taxonomy mismatch in classifier artifact."\n            )\n\n        model_id = self.manifest["model_id"]\n        model_revision = self.manifest[\n            "model_revision"\n        ]\n\n        self.processor = (\n            AutoProcessor\n            .from_pretrained(\n                self.model_dir\n                / "processor",\n                local_files_only=True,\n            )\n        )\n\n        self.processor.image_processor.size = {\n            "shortest_edge": (\n                MIN_TOKENS_PER_IMAGE\n                * (\n                    QWEN3VL_SPATIAL_COMPRESSION\n                    ** 2\n                )\n            ),\n            "longest_edge": (\n                MAX_TOKENS_PER_IMAGE\n                * (\n                    QWEN3VL_SPATIAL_COMPRESSION\n                    ** 2\n                )\n            ),\n        }\n\n        self.processor.tokenizer.padding_side = (\n            "right"\n        )\n\n        bnb_config = BitsAndBytesConfig(\n            load_in_4bit=True,\n            bnb_4bit_quant_type="nf4",\n            bnb_4bit_use_double_quant=True,\n            bnb_4bit_compute_dtype=(\n                self.compute_dtype\n            ),\n        )\n\n        base_model = (\n            AutoModelForImageTextToText\n            .from_pretrained(\n                model_id,\n                revision=model_revision,\n                quantization_config=bnb_config,\n                device_map={"": 0},\n                torch_dtype=self.compute_dtype,\n                attn_implementation=(\n                    self.attn_implementation\n                ),\n            )\n        )\n\n        base_model.config.use_cache = False\n\n        backbone = PeftModel.from_pretrained(\n            base_model,\n            self.model_dir / "adapter",\n            is_trainable=False,\n        )\n\n        hidden_size = int(\n            backbone\n            .get_base_model()\n            .config\n            .text_config\n            .hidden_size\n        )\n\n        expected_hidden_size = int(\n            head_state["hidden_size"]\n        )\n\n        if hidden_size != expected_hidden_size:\n            raise RuntimeError(\n                "Classifier hidden-size mismatch: "\n                f"{hidden_size} != "\n                f"{expected_hidden_size}"\n            )\n\n        self.model = _Qwen3VLSubjectClassifier(\n            backbone,\n            hidden_size=hidden_size,\n            num_labels=len(\n                self.subject_to_id\n            ),\n        ).to(\n            self.device\n        )\n\n        self.model.norm.load_state_dict(\n            head_state["norm"]\n        )\n\n        self.model.classifier.load_state_dict(\n            head_state["classifier"]\n        )\n\n        self.model.eval()\n\n    def _encode(\n        self,\n        *,\n        question: str,\n        images: Iterable[Any],\n        options: Iterable[str],\n    ) -> dict:\n        messages = _build_messages(\n            question=question,\n            images=images,\n            options=options,\n        )\n\n        inputs = (\n            self.processor\n            .apply_chat_template(\n                messages,\n                tokenize=True,\n                add_generation_prompt=False,\n                return_dict=True,\n                return_tensors="pt",\n            )\n        )\n\n        seq_len = int(\n            inputs[\n                "input_ids"\n            ].shape[-1]\n        )\n\n        if seq_len > MAX_SEQUENCE_TOKENS:\n            raise RuntimeError(\n                f"Classifier sequence length "\n                f"{seq_len} exceeds "\n                f"{MAX_SEQUENCE_TOKENS}; "\n                "silent truncation is disabled."\n            )\n\n        return {\n            key: (\n                value.to(\n                    self.device,\n                    non_blocking=True,\n                )\n                if torch.is_tensor(value)\n                else value\n            )\n            for key, value\n            in inputs.items()\n        }\n\n    @torch.inference_mode()\n    def predict(\n        self,\n        *,\n        question: str,\n        images: list[Image.Image],\n        options: list[str],\n    ) -> dict:\n        inputs = self._encode(\n            question=str(question),\n            images=list(images),\n            options=list(options),\n        )\n\n        with torch.autocast(\n            device_type="cuda",\n            dtype=self.compute_dtype,\n            enabled=True,\n        ):\n            logits = self.model(\n                inputs\n            )\n\n        probs = torch.softmax(\n            logits.float(),\n            dim=-1,\n        )[0]\n\n        pred_id = int(\n            torch.argmax(\n                probs\n            ).item()\n        )\n\n        subject = self.id_to_subject[\n            pred_id\n        ]\n\n        domain = self.subject_to_domain[\n            subject\n        ]\n\n        k = min(\n            3,\n            len(self.id_to_subject),\n        )\n\n        top_prob, top_idx = torch.topk(\n            probs,\n            k=k,\n        )\n\n        top3 = [\n            {\n                "subject": self.id_to_subject[\n                    int(idx)\n                ],\n                "domain": (\n                    self.subject_to_domain[\n                        self.id_to_subject[\n                            int(idx)\n                        ]\n                    ]\n                ),\n                "probability": float(prob),\n            }\n            for prob, idx in zip(\n                top_prob.detach().cpu().tolist(),\n                top_idx.detach().cpu().tolist(),\n            )\n        ]\n\n        return {\n            "subject": subject,\n            "domain": domain,\n            "confidence": float(\n                probs[\n                    pred_id\n                ].detach().cpu()\n            ),\n            "top3_subjects": top3,\n            "classifier_model": (\n                self.manifest[\n                    "model_id"\n                ]\n            ),\n            "classifier_artifact_version": (\n                ROUTER_CLASSIFIER_ARTIFACT_VERSION\n            ),\n        }\n'

compile(
    ROUTER_CLASSIFIER_SOURCE,
    "router_classifier.py",
    "exec",
)

(FINAL_DIR / "router_classifier.py").write_text(
    ROUTER_CLASSIFIER_SOURCE,
    encoding="utf-8",
)

router_contract = {
    "artifact_version": ROUTER_CLASSIFIER_ARTIFACT_VERSION,
    "entrypoint_file": "router_classifier.py",
    "entrypoint_class": "Qwen3VLRouterClassifier",
    "constructor": "Qwen3VLRouterClassifier(model_dir)",
    "predict_contract": {
        "inputs": {
            "question": "str",
            "images": "list[PIL.Image.Image]",
            "options": "list[str]",
        },
        "outputs_required": [
            "subject",
            "domain",
        ],
        "outputs_optional": [
            "confidence",
            "top3_subjects",
            "classifier_model",
            "classifier_artifact_version",
        ],
    },
    "taxonomy": {
        "n_subjects": 30,
        "domains": list(DOMAIN_TO_SUBJECTS.keys()),
    },
    "visual_representation": (
        "same interleaved question/options/image-marker policy used during training"
    ),
}

(FINAL_DIR / "router_contract.json").write_text(
    json.dumps(
        router_contract,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

(FINAL_DIR / "router_requirements.txt").write_text(
    "\n".join([
        "transformers==5.16.1",
        "peft==0.20.0",
        "bitsandbytes==0.49.0",
        "accelerate>=1.7",
        "torch",
        "Pillow>=10",
        "",
    ]),
    encoding="utf-8",
)

required_router_artifacts = [
    FINAL_DIR / "training_manifest.json",
    FINAL_DIR / "classifier_head.pt",
    FINAL_DIR / "adapter",
    FINAL_DIR / "processor",
    FINAL_DIR / "router_classifier.py",
    FINAL_DIR / "router_contract.json",
    FINAL_DIR / "router_requirements.txt",
]

missing_router_artifacts = [
    str(path)
    for path in required_router_artifacts
    if not path.exists()
]

if missing_router_artifacts:
    raise RuntimeError(
        "Router-ready export is incomplete: "
        + repr(missing_router_artifacts)
    )

print(
    "Router-ready classifier artifact: PASS |",
    FINAL_DIR,
)

Final model artifacts: /kaggle/working/mmmu_qwen3vl2b_domain_classifier/final_model
Router-ready classifier artifact: PASS | /kaggle/working/mmmu_qwen3vl2b_domain_classifier/final_model


## Router integration artifact

The `final_model/` directory is directly consumable by the adaptive router.
No hand-written bridge code is required.

The training notebook now exports:

- `adapter/` — trained QLoRA adapter;
- `processor/` — the exact multimodal processor;
- `classifier_head.pt` — 30-way subject classifier head and taxonomy;
- `training_manifest.json` — pinned model/dataset/training metadata;
- `router_classifier.py` — router-facing multimodal inference wrapper;
- `router_contract.json` — machine-readable interface contract;
- `router_requirements.txt` — runtime package requirements.

The public classifier contract is:

```python
predict(
    question=...,
    images=[...],
    options=[...],
) -> {
    "subject": "...",
    "domain": "...",
    ...
}
```

The wrapper reconstructs the same interleaved semantic representation used
during training, including question images, option images, repeated image
references, and unreferenced source images.

In [16]:
# ================================================================
# 15. Held-out-500 evaluation — resume safe
# ================================================================
model.eval()

EVAL_JSONL = WORKDIR / "heldout500_predictions.jsonl"
EVAL_CSV = WORKDIR / "heldout500_predictions.csv"

def read_jsonl(path):
    if not path.is_file():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

existing_rows = read_jsonl(EVAL_JSONL)
existing_by_id = {}

for row in existing_rows:
    sid = str(row.get("id", ""))
    if sid in existing_by_id:
        raise RuntimeError(
            f"Duplicate held-out prediction in checkpoint: {sid}"
        )
    existing_by_id[sid] = row

selected_id_set = set(selected_500_ids)
foreign_ids = sorted(set(existing_by_id) - selected_id_set)
if foreign_ids:
    raise RuntimeError(
        "Prediction checkpoint contains IDs outside the frozen 500: "
        + repr(foreign_ids[:10])
    )

print(
    "Held-out evaluation resume:",
    len(existing_by_id),
    "/ 500 already complete",
)

with torch.inference_mode():
    for eval_order, dataset_index in enumerate(
        tqdm(heldout_indices, desc="Evaluating held-out 500"),
        start=1,
    ):
        row = pro_ds[dataset_index]
        sample_id = str(row["id"])

        if sample_id in existing_by_id:
            continue

        gold_subject = str(row["subject"])

        if gold_subject not in SUBJECT_TO_ID:
            raise RuntimeError(
                f"Unexpected held-out subject: {gold_subject}"
            )

        inputs = encode_row(row)
        inputs = {
            k: v.cuda(non_blocking=True) if torch.is_tensor(v) else v
            for k, v in inputs.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=COMPUTE_DTYPE,
            enabled=True,
        ):
            logits = model(inputs)

        probs = torch.softmax(logits.float(), dim=-1)[0]
        pred_id = int(torch.argmax(probs).item())
        pred_subject = ID_TO_SUBJECT[pred_id]

        gold_domain = SUBJECT_TO_DOMAIN[gold_subject]
        pred_domain = SUBJECT_TO_DOMAIN[pred_subject]

        top3_prob, top3_idx = torch.topk(probs, k=3)
        top3 = [
            {
                "subject": ID_TO_SUBJECT[int(i)],
                "probability": float(p),
            }
            for p, i in zip(
                top3_prob.detach().cpu().tolist(),
                top3_idx.detach().cpu().tolist(),
            )
        ]

        result = {
            "evaluation_order": eval_order,
            "id": sample_id,
            "gold_subject": gold_subject,
            "pred_subject": pred_subject,
            "subject_correct": pred_subject == gold_subject,
            "gold_domain": gold_domain,
            "pred_domain": pred_domain,
            "domain_correct": pred_domain == gold_domain,
            "confidence": float(probs[pred_id].detach().cpu()),
            "top3_subjects_json": json.dumps(top3),
            "n_images": len(non_null_image_indices(row)),
        }

        with EVAL_JSONL.open("a", encoding="utf-8") as f:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")

        existing_by_id[sample_id] = result
        del inputs, logits, probs

prediction_rows = [
    existing_by_id[sid]
    for sid in selected_500_ids
    if sid in existing_by_id
]

pred_df = pd.DataFrame(prediction_rows)
pred_df.to_csv(EVAL_CSV, index=False)

if len(pred_df) == 500:
    print("Held-out evaluation COMPLETE: 500/500")
else:
    print(
        f"PARTIAL held-out evaluation: {len(pred_df)}/500. "
        "Rerun this cell to resume."
    )

print("Predictions:", EVAL_CSV)

Held-out evaluation resume: 0 / 500 already complete


Evaluating held-out 500:   0%|          | 0/500 [00:00<?, ?it/s]

Held-out evaluation COMPLETE: 500/500
Predictions: /kaggle/working/mmmu_qwen3vl2b_domain_classifier/heldout500_predictions.csv


In [17]:
# ================================================================
# 16. Final metrics and paper-ready outputs
# ================================================================
# Final paper metrics require the complete frozen cohort.
if len(pred_df) != 500:
    raise RuntimeError(
        f"Held-out evaluation is incomplete: {len(pred_df)}/500. "
        "Resume the evaluation cell before computing final metrics."
    )

subject_acc = accuracy_score(
    pred_df["gold_subject"],
    pred_df["pred_subject"],
)
subject_macro_f1 = f1_score(
    pred_df["gold_subject"],
    pred_df["pred_subject"],
    labels=SUBJECTS,
    average="macro",
)

domains = list(DOMAIN_TO_SUBJECTS.keys())

domain_acc = accuracy_score(
    pred_df["gold_domain"],
    pred_df["pred_domain"],
)
domain_macro_f1 = f1_score(
    pred_df["gold_domain"],
    pred_df["pred_domain"],
    labels=domains,
    average="macro",
)

subject_summary = (
    pred_df.groupby("gold_subject", as_index=False)
    .agg(
        n=("id", "count"),
        subject_accuracy=("subject_correct", "mean"),
        domain_accuracy=("domain_correct", "mean"),
        mean_confidence=("confidence", "mean"),
    )
)
subject_summary["subject_accuracy"] *= 100
subject_summary["domain_accuracy"] *= 100

domain_summary = (
    pred_df.groupby("gold_domain", as_index=False)
    .agg(
        n=("id", "count"),
        subject_accuracy=("subject_correct", "mean"),
        domain_accuracy=("domain_correct", "mean"),
        mean_confidence=("confidence", "mean"),
    )
)
domain_summary["subject_accuracy"] *= 100
domain_summary["domain_accuracy"] *= 100

cm = confusion_matrix(
    pred_df["gold_subject"],
    pred_df["pred_subject"],
    labels=SUBJECTS,
)
cm_df = pd.DataFrame(
    cm,
    index=[f"gold::{s}" for s in SUBJECTS],
    columns=[f"pred::{s}" for s in SUBJECTS],
)

subject_summary.to_csv(WORKDIR / "heldout500_by_subject.csv", index=False)
domain_summary.to_csv(WORKDIR / "heldout500_by_domain.csv", index=False)
cm_df.to_csv(WORKDIR / "heldout500_subject_confusion_matrix.csv")

metrics = {
    "status": "COMPLETE_HELDOUT500" if len(pred_df) == 500 else "PARTIAL",
    "n": int(len(pred_df)),
    "subject_accuracy": float(subject_acc),
    "subject_accuracy_pct": float(100 * subject_acc),
    "subject_macro_f1": float(subject_macro_f1),
    "domain_accuracy": float(domain_acc),
    "domain_accuracy_pct": float(100 * domain_acc),
    "domain_macro_f1": float(domain_macro_f1),
    "mean_confidence": float(pred_df["confidence"].mean()),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "training_rows": len(clean_records),
    "heldout500_path": str(SELECTED_500_PATH),
}

(WORKDIR / "heldout500_metrics.json").write_text(
    json.dumps(metrics, indent=2),
    encoding="utf-8",
)

print(json.dumps(metrics, indent=2))
display(domain_summary)
display(subject_summary)

{
  "status": "COMPLETE_HELDOUT500",
  "n": 500,
  "subject_accuracy": 0.886,
  "subject_accuracy_pct": 88.6,
  "subject_macro_f1": 0.8845397676596557,
  "domain_accuracy": 0.96,
  "domain_accuracy_pct": 96.0,
  "domain_macro_f1": 0.9593012298968441,
  "mean_confidence": 0.9030271834433079,
  "model_id": "Qwen/Qwen3-VL-2B-Instruct",
  "model_revision": "89644892e4d85e24eaac8bacfd4f463576704203",
  "training_rows": 8426,
  "heldout500_path": "/kaggle/input/notebooks/jingilifteyna/500-sample-mmmu-pro-held-out-test-cohort/mmmu_pro_heldout_500/selected_500_ids.txt"
}


,gold_domain,n,subject_accuracy,domain_accuracy,mean_confidence
0,Art and Design,67,80.597015,100.000000,0.842289
1,Business,84,80.952381,95.238095,0.821446
2,Health and Medicine,82,85.365854,92.682927,0.884607
3,Humanities and Social Science,67,94.029851,95.522388,0.936076
4,Science,85,90.588235,94.117647,0.937510
5,Tech and Engineering,115,96.521739,98.260870,0.966396


,gold_subject,n,subject_accuracy,domain_accuracy,mean_confidence
0,Accounting,17,82.352941,100.000000,0.822971
1,Agriculture,16,93.750000,93.750000,0.952376
2,Architecture_and_Engineering,16,100.000000,100.000000,0.997189
3,Art,17,88.235294,100.000000,0.846590
4,Art_Theory,16,93.750000,100.000000,0.861654
5,Basic_Medical_Science,17,82.352941,94.117647,0.844311
6,Biology,17,94.117647,100.000000,0.915323
7,Chemistry,17,88.235294,94.117647,0.945501
8,Clinical_Medicine,16,93.750000,100.000000,0.892645
9,Computer_Science,17,100.000000,100.000000,0.994014


## 17. Interpretation for the thesis

The primary routing metric is **domain accuracy**, because the downstream
adaptive prompt router consumes one of six domains. Subject accuracy is reported
as a stricter auxiliary metric.

Because domain is deterministically derived from the predicted subject, the
classifier can never emit a logically inconsistent subject/domain pair.

The final manuscript should report:

- exact clean-training-set size after MMMU-Pro exclusion;
- removal counts for ID, question, and visual overlap;
- subject and domain accuracy on the frozen 500;
- macro-F1 for both granularities;
- per-domain and per-subject results;
- the 30×30 subject confusion matrix;
- the frozen model/dataset revisions and QLoRA configuration.

The held-out 500 must not be used to tune epochs, LoRA rank, learning rates,
visual budgets, or class weights after observing its results.

In [18]:
# ================================================================
# 18. Final artifact inventory
# ================================================================
print("=" * 100)
print("FINAL ARTIFACTS")
print("=" * 100)

for path in sorted(WORKDIR.rglob("*")):
    if path.is_file():
        print(
            path.relative_to(WORKDIR),
            f"{path.stat().st_size / (1024**2):.2f} MiB",
        )

print("=" * 100)
print("Use heldout500_metrics.json as the primary final classifier report.")
print("=" * 100)

FINAL ARTIFACTS
audit/clean_mmmu_training_index.parquet 0.07 MiB
audit/cleaning_by_subject.csv 0.00 MiB
audit/contamination_summary.json 0.00 MiB
audit/removed_mmmu_mmmupro_exact_overlap.csv 0.23 MiB
audit/subject_class_weights.csv 0.00 MiB
checkpoints/checkpoint_step_000050/adapter/README.md 0.00 MiB
checkpoints/checkpoint_step_000050/adapter/adapter_config.json 0.00 MiB
checkpoints/checkpoint_step_000050/adapter/adapter_model.safetensors 66.56 MiB
checkpoints/checkpoint_step_000050/classifier_head.pt 0.25 MiB
checkpoints/checkpoint_step_000050/config.json 0.00 MiB
checkpoints/checkpoint_step_000050/trainer_state.pt 133.82 MiB
checkpoints/checkpoint_step_000100/adapter/README.md 0.00 MiB
checkpoints/checkpoint_step_000100/adapter/adapter_config.json 0.00 MiB
checkpoints/checkpoint_step_000100/adapter/adapter_model.safetensors 66.56 MiB
checkpoints/checkpoint_step_000100/classifier_head.pt 0.25 MiB
checkpoints/checkpoint_step_000100/config.json 0.00 MiB
checkpoints/checkpoint_step_0001